In [2]:
import sys; sys.path.append("..")   # lets the notebook import from src/
import pandas as pd
from src.data.load import load_data
from src.evaluation.nested_cv import run_nested_cv
from src.evaluation.metrics import summarize, per_fold, subject_level

X, y, groups = load_data("../data/raw/parkinsons.data")
print(X.shape, y.value_counts().to_dict(), groups.nunique())   # expect (195, 20), {1:147, 0:48}, 32

results, params = {}, {}
for name in ["logreg", "svm_rbf", "knn", "rf"]:
    
    oof, best, _ = run_nested_cv(name, X, y, groups)
    results[name], params[name] = oof, best

pd.DataFrame({n: summarize(o) for n, o in results.items()}).T.round(3)

(195, 20) {1: 147, 0: 48} 32


,accuracy,balanced_acc,sensitivity_PD,specificity_healthy,precision_PD,f1_PD,roc_auc,pr_auc_healthy
logreg,0.759,0.770,0.748,0.792,0.917,0.824,0.846,0.618
svm_rbf,0.400,0.574,0.231,0.917,0.895,0.368,0.737,0.495
knn,0.790,0.657,0.918,0.396,0.823,0.868,0.741,0.449
rf,0.703,0.627,0.776,0.479,0.820,0.797,0.759,0.481


In [3]:
for n, o in results.items():
    subj, cm, ties = subject_level(o)
    print(n); print(per_fold(o).round(3)); print(params[n]); print(cm, "ties:", ties)

logreg
                      mean    std
accuracy             0.763  0.145
balanced_acc         0.767  0.097
sensitivity_PD       0.750  0.220
specificity_healthy  0.783  0.201
precision_PD         0.920  0.084
f1_PD                0.808  0.152
roc_auc              0.852  0.131
pr_auc_healthy       0.678  0.269
   fold  inner_auc  model__C select__k
0     1   0.796225     0.001       all
1     2   0.825629    10.000         8
2     3   0.819634     0.001       all
3     4   0.857183     0.010        12
4     5   0.845951     0.001       all
[[ 7  1]
 [ 4 20]] ties: 3
svm_rbf
                      mean    std
accuracy             0.384  0.242
balanced_acc         0.578  0.126
sensitivity_PD       0.223  0.306
specificity_healthy  0.933  0.149
precision_PD         0.360  0.498
f1_PD                0.275  0.378
roc_auc              0.830  0.105
pr_auc_healthy       0.642  0.208
   fold  inner_auc  model__C  model__gamma select__k
0     1   0.795724      0.01          0.01       all
1     

In [4]:
rows = []
for s in [0, 1, 7, 21, 42]:
    oof_s, _, _ = run_nested_cv("logreg", X, y, groups, seed=s)
    rows.append({"seed": s, **summarize(oof_s)})
pd.DataFrame(rows)[["balanced_acc", "roc_auc", "specificity_healthy", "sensitivity_PD"]].describe().round(3)

,balanced_acc,roc_auc,specificity_healthy,sensitivity_PD
count,5.000,5.000,5.000,5.000
mean,0.706,0.779,0.692,0.721
std,0.046,0.069,0.092,0.035
min,0.648,0.667,0.542,0.667
25%,0.677,0.765,0.688,0.714
50%,0.711,0.804,0.708,0.721
75%,0.725,0.812,0.729,0.748
max,0.770,0.846,0.792,0.755


In [5]:
_, _, imp = run_nested_cv("logreg", X, y, groups, importance=True)
imp.assign(mean=imp.mean(axis=1), std=imp.std(axis=1))[["mean", "std"]].sort_values("mean", ascending=False).round(3)

,mean,std
spread1,0.038,0.046
PPE,0.026,0.025
spread2,0.012,0.031
MDVP:APQ,0.011,0.023
Shimmer:APQ5,0.002,0.009
MDVP:Shimmer(dB),-0.001,0.009
MDVP:Shimmer,-0.002,0.006
MDVP:PPQ,-0.002,0.009
D2,-0.003,0.023
DFA,-0.003,0.019


In [6]:
from src.models.train_final import train_final
final_search = train_final("logreg", X, y, groups)
final = final_search.best_estimator_
print(final_search.best_params_)
coefs = pd.Series(final.named_steps["model"].coef_[0],
                  index=final[:-1].get_feature_names_out()).sort_values()
coefs.round(3)

{'model__C': 0.001, 'select__k': 'all'}


HNR                -0.023
MDVP:Flo(Hz)       -0.018
MDVP:Fo(Hz)        -0.018
MDVP:Fhi(Hz)       -0.013
DFA                 0.015
RPDE                0.017
NHR                 0.020
Shimmer:APQ3        0.021
Shimmer:APQ5        0.022
MDVP:RAP            0.023
D2                  0.023
MDVP:Jitter(%)      0.023
MDVP:Jitter(Abs)    0.023
MDVP:Shimmer(dB)    0.023
MDVP:Shimmer        0.023
MDVP:PPQ            0.024
MDVP:APQ            0.027
spread2             0.035
PPE                 0.040
spread1             0.042
dtype: float64